In [3]:
from datasets import load_dataset
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
import regex as re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Downloading the dataset

In [2]:
dataset = load_dataset("imdb")

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 303988.38 examples/s]


# Data Representation

## A. Sparse Features

- Includes BoW and TF-IDF feature extraction
- We have included our own implementation of certain key functions to show clarity in concepts being used. However, we will be using efficient pre-defined functions on our dataset, since it is large

In [ ]:
def tokenize(text, mode='unigram'):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS]

    if mode == 'unigram':
        return tokens

    elif mode == 'bigram':
        bigrams = [tokens[i] + "_" + tokens[i+1] for i in range(len(tokens)-1)]
        return bigrams

    elif mode == 'both':
        bigrams = [tokens[i] + "_" + tokens[i+1] for i in range(len(tokens)-1)]
        return tokens + bigrams
    else:
       raise ValueError("mode is not in {'unigram', 'bigram', 'both'}")


def bowdict(docs):
  d = {}
  x = 0
  for i in docs:
    i = tokenize(i)
    for j in i:
      if j not in d:
        d[j] = x
        x+= 1
  return d

def bow_vectorize(doc, vocab):
    vec = np.zeros(len(vocab))
    for word in tokenize(doc):
        if word in vocab:
          vec[vocab[word]] += 1

    return vec

def bow_matrix(docs):
    vocab = bowdict(docs)
    matrix = np.array([bow_vectorize(doc, vocab) for doc in docs])
    return matrix, vocab

In [ ]:
def tf(bowed):
    row_sums = bowed.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    return bowed / row_sums

def idf(bowed):
    N = bowed.shape[0]
    df = (bowed > 0).sum(axis=0)
    return np.log((N + 1) / (df + 1)) + 1


def build_index(docs):
    bowed, vocab = bow_matrix(docs)
    tf_bowed = tf(bowed)
    idf_vec = idf(bowed)
    tfidf_docs = tf_bowed * idf_vec
    norms = np.linalg.norm(tfidf_docs, axis=1, keepdims=True)
    tfidf_docs = tfidf_docs / norms
    return tfidf_docs, vocab, idf_vec

In [ ]:
def vectorize_query(query, vocab, idf_vec):
    vec = np.zeros(len(vocab))
    for word in tokenize(query):
        if word in vocab:
            vec[vocab[word]] += 1

    if vec.sum() == 0:
        return vec
    tf_vec = vec / vec.sum()
    return tf_vec * idf_vec

def cosine_similarity(tfidf_docs, query_vec):
    dot_products = tfidf_docs @ query_vec
    doc_norms = np.linalg.norm(tfidf_docs, axis=1)
    query_norm = np.linalg.norm(query_vec)

    if query_norm == 0:
        return np.zeros(len(tfidf_docs))

    similarities = dot_products / (doc_norms * query_norm)
    return similarities